# Tugas 01 : **Crawling Berita**

NAMA : Mohammad Hasan Basri

NIM  : 210411100169

MATA KULIAH : Pencarian dan Penambangan Web - A

# Tugas Crawling Berita Online Untuk Mendapatkan **Judul**, **Tanggal**, **Isi** dan **Kategori Berita** dari sebuah halaman website berita online

**Pengertian Crawling**


Crawling adalah proses otomatisasi yang dilakukan oleh program komputer untuk menjelajahi dan mengumpulkan data dari halaman-halaman web di internet. Proses ini sering kali dilakukan oleh bot yang dikenal sebagai web crawlers atau spiders. Web crawlers ini akan menelusuri (crawl) berbagai situs web, mengakses halaman-halaman yang ada, dan mengunduh atau mengekstraksi informasi yang dibutuhkan untuk kemudian disimpan atau diindeks dalam database.

**Crawling website berita adalah** *proses otomatis mengumpulkan data dari berbagai situs berita di internet. Proses ini dilakukan oleh program khusus yang disebut crawler atau spider. Crawler ini akan menjelajahi internet, mengunjungi berbagai situs berita, dan mengambil data seperti judul berita, isi berita, tanggal publikasi, dan tautan terkait.*

**Kegunaan BeautifulSoup**

**Web Scraping:** *Mengambil data dari halaman web untuk berbagai tujuan, seperti analisis sentimen, riset pasar, dan pengembangan aplikasi.*

**Parsing Data:** *Mengubah data yang tidak terstruktur menjadi format yang terstruktur.*

**Automasi:** *Mengotomatiskan tugas-tugas yang berulang, seperti mengunduh data dari banyak halaman web.*


# Code Program Proses **Crawling Berita Online :**

Dalam proses crawling ini saya menggunakan beberapa library dan diantara library yang penting di import adalah *BeautifulSoup* yang berfungsi sebagai library crawler

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

Membuat fungsi filter konten
Berfungsi untuk Mem-filter dari elemen-elemen HTML pada berita yang tidak diinginkan , Contoh kasus seperti iklan, daftar isi, gambar , link sisipan, dll

In [2]:

# Fungsi untuk membersihkan konten dari elemen-elemen yang tidak diinginkan
def clean_content(content_element):
    if content_element:
        # Hapus elemen yang berisi daftar isi
        for daftar_isi in ["collapsible"]:
            unwanted = content_element.find("div", id=daftar_isi)
            if unwanted:
                unwanted.decompose()

        # Hapus elemen yang berisi tag
        for tag_class in ["aevp", "detail__body-tag mgt-16"]:
            unwanted = content_element.find_all("div", class_=tag_class)
            for el in unwanted:
                el.decompose()

        # Hapus elemen yang berisi link sisipan
        link_sisip = content_element.find_all("table", class_="linksisip")
        for table in link_sisip:
            table.decompose()

        # Hapus elemen paragraf dan span dengan class 'para_caption'
        unwanted_paragraphs = content_element.find_all(["p", "span"], class_="para_caption")
        for para in unwanted_paragraphs:
            para.decompose()

        # Kembalikan teks yang tersisa
        return content_element.get_text(separator=' ', strip=True).strip()

    return "Content Not Found"


Membuat fungsi untuk melakukan crawling data pada situs web Detik.com.
Fungsi ini mengambil data berupa judul berita, tanggal publikasi, isi berita dan kategori berita yang terdapat di halaman tersebut.


In [ ]:
# Fungsi untuk mengambil data dari halaman web Detik.com
def get_data(url, kategori, min_articles_per_category):
    try:
        response = requests.get(url)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")
        return

    soup = BeautifulSoup(response.content, "html.parser")
    articles = soup.find_all("article", class_="list-content__item")

    for article in articles:
        if len([k for k in kategori_list if k == kategori]) >= min_articles_per_category:
            return  # Menghentikan proses jika jumlah artikel sudah mencapai minimum yang diinginkan

        try:
            link = article.find("a")["href"]
            article_response = requests.get(link)
            article_response.raise_for_status()
        except (requests.exceptions.RequestException, TypeError) as e:
            print(f"Request for article failed: {e}")
            continue

        article_soup = BeautifulSoup(article_response.content, "html.parser")
        title_element = article_soup.find("h1", class_="detail__title")
        title = title_element.text.strip() if title_element else "Title Not Found"
        date_element = article_soup.find("div", class_="detail__date")
        date = date_element.text.strip() if date_element else "Date Not Found"
        content_element = article_soup.find("div", class_="detail__body-text")
        content = content_element.text.strip() if content_element else "Content Not Found"

        # Bersihkan konten menggunakan fungsi clean_content
        content = clean_content(content_element)

        judul.append(title)
        tanggal.append(date)
        isi.append(content)
        kategori_list.append(kategori)

        if len(judul) <= 100:
            print(title)
        time.sleep(1)

# Membuat list url dan kategori yang akan di-crawl
base_urls = [
    "https://travel.detik.com/travel-news/indeks",
    "https://www.detik.com/hikmah/indeks",
]
categories = [
    "Pariwisata",
    "Keislaman",
]

# Inisialisasi list untuk menyimpan data
judul = []
tanggal = []
isi = []
kategori_list = []

# Batas minimal artikel per kategori
min_articles_per_category = 50

# Melakukan iterasi untuk setiap url dan kategori
for base_url, category in zip(base_urls, categories):
    page = 1
    while len([k for k in kategori_list if k == category]) < min_articles_per_category:
        url = f"{base_url}/{page}"
        get_data(url, category, min_articles_per_category)
        time.sleep(2)
        page += 1

# Membuat dataframe dari list data
df = pd.DataFrame({"judul": judul, "tanggal": tanggal, "isi": isi, "kategori": kategori_list})

# Menyimpan dataframe ke file csv
df.to_csv("Crawl-berita-Pariwisata&Keislaman.csv", index=False)

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import pandas as pd
# Read Excel file
df = pd.read_excel("/content/drive/My Drive/PPWA/report/Tugas_PPWA/Crawl-berita-Pariwisata&Keislaman.xlsx")
df.head(100)

,judul,tanggal,isi,kategori
0,Iran Serukan Negara Muslim Bersatu Hentikan Ke...,"Kamis, 17 Okt 2024 10:01 WIB",Jakarta - Presiden Iran Masoud Pezeshkian mene...,Keislaman
1,"Bacaan Al-Qur'an Surah Asy-Syu'ara: Arab, Lati...","Kamis, 17 Okt 2024 09:30 WIB",NaN,Keislaman
2,"Rukun Iman Ke-2, Setiap Muslim Wajib Mengimani...","Kamis, 17 Okt 2024 08:45 WIB",Jakarta - Malaikat adalah makhluk ciptaan Alla...,Keislaman
3,Kalender Ramadhan 2025 Muhammadiyah dan Predik...,"Kamis, 17 Okt 2024 08:00 WIB",Jakarta - Kalender Ramadhan 2025 versi Muhamma...,Keislaman
4,20 Gambaran Kehidupan di Neraka Menurut Al-Qur...,"Kamis, 17 Okt 2024 07:15 WIB",Jakarta - Gambaran kehidupan di neraka selalu ...,Keislaman
...,...,...,...,...
95,"Praha Incar Turis-Turis Kaya, 'Wisata Pub Jala...","Rabu, 16 Okt 2024 06:13 WIB",Praha - Praha ingin mengganti imej dari kota s...,Pariwisata
96,"Usai Festival Vegetarian, Phuket Pusing dengan...","Rabu, 16 Okt 2024 05:39 WIB",Phuket - Festival Vegetarian menjadi salah sat...,Pariwisata
97,Ngeri! Turis Inggris Tewas Saat Memanjat Jemba...,"Rabu, 16 Okt 2024 05:01 WIB",Jakarta - Seorang turis yang juga kreator kont...,Pariwisata
98,Info Loker: Lion Group Buka Lowongan Mekanik Nih!,"Selasa, 15 Okt 2024 23:04 WIB",Jakarta - Traveler yang ingin bekerja di dunia...,Pariwisata
